In [1]:
from train_stock_factor import *
from vnpy_portfoliostrategy import StrategyTemplate
from vnpy_portfoliostrategy.backtesting import BacktestingEngine
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib

train_start=datetime(2025, 4, 1)
train_end=datetime(2025, 10, 1)
test_start=datetime(2025, 10, 2)
test_end=datetime(2026, 1, 1)

kind_args=[1,1]
factors_dir='./all_result_1_1'

stocks=get_stock_labels('stock_zz1000')
stocks=np.array(stocks)[:20]
sub_stocks=stocks[[int((len(stocks)-1)/20*i) for i in range(20)]]

In [2]:
repeat=set()
factors=[]
for path in os.listdir(factors_dir):
    with open(factors_dir+'/'+path, "r") as f:
        pop_data = json.load(f)
        for i in pop_data:
            t=tree()
            t.load(i)
            t.update_depth()
            if t.express() not in repeat:
                factors.append(t)
                repeat.add(t.express())
#renew fitness ?
factors=sorted(factors,key=lambda x:-1*x.fitness)[1000:1100]
tree_dir='tree_dir'
if os.path.exists(tree_dir):
    shutil.rmtree(tree_dir)
os.makedirs(tree_dir, exist_ok=True)
for i in range(len(factors)):
    with open(tree_dir+'/'+str(i)+".json", "w") as f:
        json.dump(factors[i].save(), f)

In [11]:
training_data=[]
for stock in stocks:
    stock_data={}
    index=0
    for factor in factors:
        content=fitnessForstock(factor.save(),train_start,train_end,stock,"fourdays_avg_std")
        for f,price,date in content:
            if date not in stock_data.keys():
                stock_data[date]=[None for i in range(len(factors)+1)]
            stock_data[date][index]=f
            stock_data[date][len(factors)]=price
        index+=1
    stock_data=np.array(list(stock_data.values()), dtype=float).tolist()
    training_data+=stock_data
training_data=np.array(training_data)

np.set_printoptions(suppress=True)
print(np.round(training_data[:10,-5:],3))

test_data=[]
for stock in stocks:
    stock_data={}
    index=0
    for factor in factors:
        content=fitnessForstock(factor.save(),test_start,test_end,stock,"fourdays_avg_std")
        for f,price,date in content:
            if date not in stock_data.keys():
                stock_data[date]=[None for i in range(len(factors)+1)]
            stock_data[date][index]=f
            stock_data[date][len(factors)]=price
        index+=1
    stock_data=np.array(list(stock_data.values()), dtype=float).tolist()
    test_data+=stock_data
test_data=np.array(test_data)

X_train = training_data[:,:-1]
y_train = training_data[:,len(training_data[0])-1]

X_test = test_data[:,:-1]
y_test = test_data[:,len(test_data[0])-1]

model_path='model'
model = LinearRegression()
model.fit(X_train, y_train)
joblib.dump(model, model_path+".pkl")
y_pred = model.predict(X_test)

# 评价指标
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))
print("y_test mean:", y_test.mean())
print("y_pred mean:", y_pred.mean())
print("y_test std:", y_test.std())
print("y_pred std:", y_pred.std())
correct = np.sign(y_test-1) == np.sign(y_pred-1)
accuracy = correct.mean()
print("方向准确率:", accuracy)

[[25500.009 25500.009 25500.035 51000.009     1.002]
 [25500.009 25500.009 25500.035 51000.009     0.995]
 [25500.009 25500.009 25500.035 51000.009     0.997]
 [25500.009 25500.009 25500.035 51000.009     0.997]
 [25500.01  25500.01  25500.036 51000.01      0.996]
 [25500.01  25500.01  25500.036 51000.01      0.995]
 [25500.01  25500.01  25500.036 51000.01      0.987]
 [25500.01  25500.01  25500.036 51000.01      0.985]
 [25500.01  25500.01  25500.036 51000.01      0.988]
 [25500.006 25500.006 25500.032 51000.006     1.   ]]
MSE: 0.0008421067731210145
R2: -0.2914501881826017
y_test mean: 0.9870666828610514
y_pred mean: 0.990603079944456
y_test std: 0.025535522593041784
y_pred std: 0.013519730645644103
方向准确率: 0.7086956521739131


In [ ]:
# X_train={}
# Y_train={}
# for i in range(len(stocks)):
#     stock=stocks[i]
#     stock_data={}
#     index=0
#     for factor in factors:
#         content=fitnessForstock(factor.save(),train_start,train_end,stock,kind_args)
#         for f,price,date in content:
#             if date not in stock_data.keys():
#                 stock_data[date]=[None for i in range(len(factors)+1)]
#             stock_data[date][index]=f
#             stock_data[date][len(factors)]=price
#         index+=1
#     for date in stock_data.keys():
#         if date not in X_train.keys():
#             X_train[date]=[None for i in range(len(stocks))]
#             Y_train[date]=[None for i in range(len(stocks))]
#         X_train[date][i]=stock_data[date][:len(factors)]
#         Y_train[date][i]=stock_data[date][len(factors)]

# x_train_data=[]
# y_train_data=[]
# for date in X_train.keys():
#     if None in Y_train[date]:
#         continue
#     x_train_data.append(X_train[date])
#     y_train_data.append(Y_train[date])
# x_train_data=np.array(x_train_data)
# y_train_data=np.array(y_train_data)

# np.set_printoptions(suppress=True)
# print(np.round(x_train_data[:10,-5:],3))

# X_test={}
# Y_test={}
# for i in range(len(stocks)):
#     stock=stocks[i]
#     stock_data={}
#     index=0
#     for factor in factors:
#         content=fitnessForstock(factor.save(),test_start,test_end,stock,kind_args)
#         for f,price,date in content:
#             if date not in stock_data.keys():
#                 stock_data[date]=[None for i in range(len(factors)+1)]
#             stock_data[date][index]=f
#             stock_data[date][len(factors)]=price
#         index+=1
#     for date in stock_data.keys():
#         if date not in X_test.keys():
#             X_test[date]=[None for i in range(len(stocks))]
#             Y_test[date]=[None for i in range(len(stocks))]
#         X_test[date][i]=stock_data[date][:len(factors)]
#         Y_test[date][i]=stock_data[date][len(factors)]

# x_test_data=[]
# y_test_data=[]
# for date in X_test.keys():
#     if None in Y_test[date]:
#         continue
#     x_test_data.append(X_test[date])
#     y_test_data.append(Y_test[date])
# x_test_data=np.array(x_test_data)
# y_test_data=np.array(y_test_data)


[[[25500.013 25500.013 25500.013 ... 25500.008 25500.033 51000.008]
  [25500.014 25500.014 25500.015 ... 25500.01  25500.036 51000.01 ]
  [25500.014 25500.014 25500.014 ... 25500.009 25500.035 51000.009]
  [25500.014 25500.014 25500.014 ... 25500.009 25500.036 51000.009]
  [25500.013 25500.013 25500.014 ... 25500.009 25500.034 51000.009]]

 [[25500.013 25500.013 25500.013 ... 25500.008 25500.033 51000.008]
  [25500.014 25500.014 25500.015 ... 25500.01  25500.036 51000.01 ]
  [25500.014 25500.014 25500.014 ... 25500.009 25500.035 51000.009]
  [25500.014 25500.014 25500.014 ... 25500.009 25500.035 51000.009]
  [25500.014 25500.014 25500.015 ... 25500.01  25500.036 51000.01 ]]

 [[25500.013 25500.013 25500.013 ... 25500.008 25500.033 51000.008]
  [25500.014 25500.014 25500.015 ... 25500.01  25500.036 51000.01 ]
  [25500.014 25500.014 25500.014 ... 25500.009 25500.035 51000.009]
  [25500.014 25500.014 25500.014 ... 25500.009 25500.035 51000.009]
  [25500.014 25500.014 25500.015 ... 25500.0

In [ ]:
# import torch
# from torch.utils.data import Dataset
# from trainDL.model.model import TradeModel
# from trainDL.trainer import Trainer, TrainingArguments

# class TradeDataset(Dataset):
#     def __init__(self, x,y):
#         self.x      = torch.tensor(x,dtype=torch.float32)
#         self.labels = torch.tensor(y,dtype=torch.float32)
#         #shuffle stock
#         #输出weight 全为正，应该剔除部分股票
#     def __len__(self): return len(self.x)
#     def __getitem__(self, i): return {"x": self.x[i], "labels": self.labels[i]}

# model = TradeModel(
#     d_model=64, n_heads=4, n_layers=3,
#     factor_size=len(x_train_data[0][0]),
#     output_mode="raw",
#     loss_type="ranking",
# )

# args = TrainingArguments(
#     output_dir="output",
#     num_epochs=20,
#     per_device_batch_size=32,
#     learning_rate=1e-4,
#     warmup_ratio=0.05,
#     eval_strategy="epoch",
#     save_strategy="epoch",
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=TradeDataset(x_train_data,y_train_data),
#     eval_dataset=TradeDataset(x_test_data,y_test_data),
# )

# state = trainer.train()

# model_path="model"
# torch.save(model.state_dict(), model_path+".pth" )


# logits = trainer.predict(TradeDataset(x_test_data,y_test_data))
# print(f"推理输出 shape: {logits.shape}")
# print(np.average(logits))
# print(np.average(y_test_data))



2026-03-11 13:11:05 | INFO | 开始训练 | epochs=20  steps/epoch=4  total_steps=80  device=cuda
2026-03-11 13:11:05 | INFO | ── epoch 1 完成  avg_loss=2.9957  耗时=0.5s
2026-03-11 13:11:05 | INFO | eval_loss=2.9957
2026-03-11 13:11:05 | INFO | checkpoint 已保存 → output\epoch-1\model.pt
2026-03-11 13:11:05 | INFO | ── epoch 2 完成  avg_loss=2.9957  耗时=0.0s
2026-03-11 13:11:05 | INFO | eval_loss=2.9957
2026-03-11 13:11:05 | INFO | checkpoint 已保存 → output\epoch-2\model.pt
2026-03-11 13:11:05 | INFO | ── epoch 3 完成  avg_loss=2.9957  耗时=0.0s
2026-03-11 13:11:05 | INFO | eval_loss=2.9957
2026-03-11 13:11:05 | INFO | checkpoint 已保存 → output\epoch-3\model.pt
2026-03-11 13:11:05 | INFO | ── epoch 4 完成  avg_loss=2.9957  耗时=0.0s
2026-03-11 13:11:05 | INFO | eval_loss=2.9957
2026-03-11 13:11:05 | INFO | checkpoint 已保存 → output\epoch-4\model.pt
2026-03-11 13:11:05 | INFO | epoch 5/20  step 20/80  loss=2.9957  avg=2.9957  lr=8.96e-05
2026-03-11 13:11:05 | INFO | ── epoch 5 完成  avg_loss=2.9957  耗时=0.0s
2026-03-11 

推理输出 shape: torch.Size([46, 20])
0.27982426
0.9870666828610514


In [ ]:
class CrossSectionStrategy(StrategyTemplate):

    parameters = ["tree_dir","model"]
    def __init__(self, strategy_engine, strategy_name, vt_symbols, setting):
        super().__init__(strategy_engine, strategy_name, vt_symbols, setting)
        self.vt_symbols=vt_symbols
        self.ams = {}
        for vt_symbol in vt_symbols:
            self.ams[vt_symbol] = ArrayManager(15)
        self.factors=[]
        for i in range(len(os.listdir(setting["tree_dir"]))):
            with open(setting["tree_dir"]+'/'+str(i)+'.json', "r") as f:
                t=tree()
                t.load(json.load(f))
                t.update_depth()
            self.factors.append(t)
       
        self.model=joblib.load(setting["model"]+".pkl")

        # 重新创建模型实例
        # self.model = TradeModel(
        #     d_model=64, n_heads=4, n_layers=3,
        #     factor_size=len(self.factors),
        #     output_mode="long_only",
        #     loss_type="ranking",
        # )
        # self.model.load_state_dict(torch.load(setting["model"]+".pth"))
        # self.model.eval()

        self.account=self.strategy_engine.capital
        self.poss={vt_symbol:0 for vt_symbol in self.vt_symbols}
        self.targets={vt_symbol:0 for vt_symbol in self.vt_symbols}

        self.features={}
        self.dfs={}
        for vt_symbol in vt_symbols:
            self.features[vt_symbol]={}
            for symbol,exchange,file in get_stock_labels('stock_zz1000_inda'):
                if vt_symbol==f"{symbol}.{exchange.value}":
                    df=pd.read_csv('stock_zz1000_inda/'+file)
                    df = df.drop(['ts_code','pe_ttm', 'dv_ttm', 'pe', 'dv_ratio','datetime'], axis=1)
                    df = df.sort_values("trade_date")
                    df = df.set_index("trade_date")
                    self.features[vt_symbol]={}
                    for i in df.columns.tolist():
                        self.features[vt_symbol][i]=[]
                    df["free_share"] = df["free_share"].ffill()
                    df["turnover_rate_f"] = df["turnover_rate_f"].ffill()
                    df["pb"] = df["pb"].ffill()
                    self.dfs[vt_symbol]=df
                    break
    
    def on_init(self):
        self.load_bars(15)
    
    def on_bars(self, bars):
        """
        bars: dict {vt_symbol: BarData}
        """

        for stock, bar in bars.items():
            self.ams[stock].update_bar(bar)
            if int(bar.datetime.date().strftime("%Y%m%d")) in self.dfs[stock].index:
                factor = self.dfs[stock].loc[int(bar.datetime.date().strftime("%Y%m%d"))]
                for i in self.features[stock].keys():
                    self.features[stock][i].append(factor[i].item())

        for stock, bar in bars.items():
            if not self.ams[stock].inited:
                return


        #买入未来四天的
        #-5为当天

        for stock in self.vt_symbols:
            pos = self.poss[stock]
            if pos > 0:
                self.sell(stock, self.ams[stock].close[-5:][0], pos/np.ceil(self.targets[stock]/4))
                self.poss[stock]-=pos/np.ceil(self.targets[stock]/4)
                self.put_event()
                self.account+=self.ams[stock].close[-5:][0]*pos/np.ceil(self.targets[stock]/4)*self.strategy_engine.sizes[stock]
                print('sell',stock,pos/np.ceil(self.targets[stock]/4),self.ams[stock].close[-5:][0],self.ams[stock].close[-5:][0]*pos/np.ceil(self.targets[stock]/4)*self.strategy_engine.sizes[stock])
                self.targets[stock]-=1
        print(1,self.account)


        #线性回归+等权买入
        #-5为当天

        data=[]
        for stock, bar in bars.items():
            if int(bar.datetime.date().strftime("%Y%m%d")) not in self.dfs[stock].index:
                continue
            close_array = self.ams[stock].close
            input={'price':np.array(close_array[:-5])}
            for i in self.features[stock].keys():
                input[i]=np.array(self.features[stock][i][-15:-5])
            x=[]
            for factor in self.factors:
                result=factor.forward(input)
                x.append(result)
            y = self.model.predict([x])[0]
            data.append((stock,y))
        

        data=[x for x in data if x[1] > 0]
        data=sorted(data,key=lambda x:-1*x[1])[:int(len(data)/5)]
        num=self.account / len(data)
        for stock,result in data:
            self.targets[stock]+=4
            price=self.ams[stock].close[-5:][0]
            volume = num/ price / self.strategy_engine.sizes[stock]
            self.buy(stock, price, volume)
            self.poss[stock]+=volume
            self.put_event()
            self.account-=price*volume*self.strategy_engine.sizes[stock]
            print('buy',stock,volume,price,price*volume*self.strategy_engine.sizes[stock])
        print(2,np.round(self.account))


        #DL+ranking买入
        #-5为当天
        # x=[]
        # used_stocks=[]
        # for stock, bar in bars.items():
        #     if int(bar.datetime.date().strftime("%Y%m%d")) not in self.dfs[stock].index:
        #         continue
        #     used_stocks.append(stock)

        #     close_array = self.ams[stock].close
        #     input={'price':np.array(close_array[:-5])}
        #     for i in self.features[stock].keys():
        #         input[i]=np.array(self.features[stock][i][-15:-5])
        #     tem_x=[]
        #     for factor in self.factors:
        #         result=factor.forward(input)
        #         tem_x.append(result)
        #     x.append(tem_x)
        # y = trainer.predict(TradeDataset([x],[0]))[0].numpy()
        # data=[]
        # for i in range(len(used_stocks)):
        #     data.append((used_stocks[i],y[i]))

        # data=sorted(data,key=lambda x:x[1])[:int(len(data)/5)]
        # sum=0
        # for stock,result in data:
        #     sum+=1/result
        # tem_account=self.account
        # for stock,result in data:
        #     price=self.ams[stock].close[-5:][0]
        #     self.targets[stock]+=4
        #     volume = tem_account*1/result/sum/price / self.strategy_engine.sizes[stock]
        #     self.buy(stock, price, volume)
        #     self.poss[stock]+=volume
        #     self.put_event()
        #     self.account-=price*volume*self.strategy_engine.sizes[stock]
        #     print('buy',stock,volume,price,price*volume*self.strategy_engine.sizes[stock])
        # print(2,np.round(self.account))


In [23]:
from datetime import datetime

from vnpy.trader.constant import Interval
from vnpy_portfoliostrategy.backtesting import BacktestingEngine



n = len(stocks)

engine = BacktestingEngine()


tem_stocks=[f"{symbol}.{exchange.value}" for symbol,exchange,file in stocks]
engine.set_parameters(
    vt_symbols=tem_stocks,
    interval=Interval.DAILY,
    start=test_start,
    end=test_end,
    rates={s:0.00 for s in tem_stocks},
    slippages={s:0.0 for s in tem_stocks},
    sizes={s:1 for s in tem_stocks},
    priceticks={s:0.01 for s in tem_stocks},
    capital=1_000_000
)

engine.add_strategy(CrossSectionStrategy, {"tree_dir":tree_dir,"model":model_path})

engine.load_data()

engine.run_backtesting()

engine.calculate_result()
engine.calculate_statistics()

engine.show_chart()

2026-03-16 17:51:17.815993	开始加载历史数据
2026-03-16 17:51:17.815993	000012.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000019.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000028.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000029.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000030.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000035.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000048.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000049.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000058.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000059.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000061.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000065.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000089.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000099.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000156.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000403.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.815993	000420.SZSE历史数据加载完成，数据量：60
2026-03-16 17:51:17.816994	000422.SZSE历史数据加载完成